In [1]:
#Importing libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split


In [5]:
df = pd.read_csv('../data/processed/loan_data_cleaned.csv')

In [6]:
X = df.drop("Loan_Approved", axis=1)
y = df["Loan_Approved"]

# X
# y

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,y,test_size=0.2, random_state=42)


# X_train
# y_train

In [8]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled
X_test_scaled

array([[ 0.42610546,  0.55091079,  1.19037073, ..., -0.41388776,
         1.16316   , -0.30723158],
       [ 1.00971366,  0.58995935,  0.18392964, ..., -0.41388776,
         1.16316   , -0.30723158],
       [-0.67557096, -0.54558674,  1.09887608, ..., -0.41388776,
         1.16316   , -0.30723158],
       ...,
       [-0.67980585, -0.10106072,  1.28186537, ..., -0.41388776,
         1.16316   , -0.30723158],
       [-0.37650705, -0.97825874, -0.54802752, ..., -0.41388776,
        -0.85972695, -0.30723158],
       [-0.73647272, -1.24497436,  0.73289751, ..., -0.41388776,
         1.16316   , -0.30723158]], shape=(200, 27))

In [10]:
#Model train
# from sklearn.linear_model import LogisticRegression
# from sklearn.metrics import recall_score, precision_score, f1_score, confusion_matrix, accuracy_score

# log_model = LogisticRegression()
# log_model.fit(X_train_scaled, y_train)

# y_pred = log_model.predict(X_test_scaled)
# y_pred_train = log_model.predict(X_train_scaled)

# print("Logistic model")
# print("Precision Score: ",precision_score(y_test, y_pred))
# print("Recall Score: ",recall_score(y_test, y_pred))
# print("F1 Score: ",f1_score(y_test, y_pred))
# print("Accuracy Score: ",accuracy_score(y_test, y_pred))
# print("Accuracy Score train: ",accuracy_score(y_train, y_pred_train))
# print("Confusion_matrix : ",confusion_matrix(y_test, y_pred))


# #Improving accuracy

# #Random forest
# from sklearn.ensemble import RandomForestClassifier

# rf = RandomForestClassifier(
#     class_weight= 'balanced',
#     n_estimators=401,
#     oob_score=True,
#     max_depth=4
    
# )

# rf.fit(X_train_scaled, y_train)

# y_pred = rf.predict(X_test_scaled)
# y_pred_train = log_model.predict(X_train_scaled)

# print("OOB score :", rf.oob_score_ *100)
# print("Accurcy score:",accuracy_score(y_test, y_pred)*100)

# print("Random forest model")
# print("Precision Score: ",precision_score(y_test, y_pred))
# print("Recall Score: ",recall_score(y_test, y_pred))
# print("F1 Score: ",f1_score(y_test, y_pred))
# print("Accuracy Score: ",accuracy_score(y_test, y_pred))
# print("Accuracy Score train: ",accuracy_score(y_train, y_pred_train))
# print("Confusion_matrix : ",confusion_matrix(y_test, y_pred))

# from sklearn.svm import SVC
# model = SVC(kernel='rbf')

# model.fit(X_train, y_train)
# y_pred = model.predict(X_test)
# accuracy_score(y_test, y_pred)


# from sklearn.ensemble import  VotingClassifier

# from sklearn.svm import SVC
# from sklearn.linear_model import LogisticRegression

# rf = RandomForestClassifier(
#     class_weight= 'balanced',
#     n_estimators=401,
#     oob_score=True,
#     max_depth=4   
# )

# SVC = SVC(probability=True)

# lr = LogisticRegression(max_iter=10000)

# voting_model = VotingClassifier(estimators=[
#     ('lr', lr),
#     ('svc', SVC),   
#     ('rfc', rf),
# ], voting='soft')
# voting_model.fit(X_train_scaled, y_train)
# y_pred = voting_model.predict(X_test_scaled)

# accuracy_score(y_test, y_pred)

# import xgboost as xgb
# from sklearn.metrics import classification_report
# xgb_clf = xgb.XGBClassifier(
#     n_estimators=400,
#     max_depth=4,
#     learning_rate=0.1,
#     eval_metric="logloss",
#     random_state=42
# )

# xgb_clf.fit(X_train_scaled, y_train)

# y_pred = xgb_clf.predict(X_test_scaled)

# print("Acc",accuracy_score(y_test, y_pred)*100, "%")
# print("Clf report",classification_report(y_test, y_pred))

# from catboost import CatBoostClassifier

# cat = CatBoostClassifier(
#     iterations=200,
#     learning_rate=0.05,
#     depth=6,
#     auto_class_weights='Balanced',  # handles imbalance automatically
#     verbose=0,                       # silences training logs
#     random_state=42
# )

# cat.fit(X_train_scaled, y_train)
# y_pred = cat.predict(X_test_scaled)
# print("Accuracy score",accuracy_score(y_test, y_pred)*100,"%")

In [11]:
#Stacking classifier
from sklearn.ensemble import StackingClassifier
import xgboost as xgb
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

lr = LogisticRegression(max_iter=5000)

rf = RandomForestClassifier(
    class_weight= 'balanced',
    n_estimators=400,
    oob_score=True,
    max_depth=4,
    random_state=42
)

xgb_clf = xgb.XGBClassifier(
    n_estimators=400,
    max_depth=4,
    learning_rate=0.1,
    eval_metric="logloss",
    random_state=42
)

final_model = LogisticRegression(max_iter=5000)

stacking_model = StackingClassifier(estimators=[
        ('lr', lr),
        ('rf',rf),
        ("xgb",xgb_clf)
        ],
        final_estimator = final_model,
        cv=5
)

stacking_model.fit(X_train_scaled, y_train)
y_pred = stacking_model.predict(X_test_scaled)

from sklearn.metrics import classification_report, accuracy_score

print("Accuracy score: ",accuracy_score(y_test, y_pred)*100)
print("Classification Report:\n ",classification_report(y_test, y_pred))

Accuracy score:  93.5
Classification Report:
                precision    recall  f1-score   support

           0       0.98      0.93      0.95       139
           1       0.85      0.95      0.90        61

    accuracy                           0.94       200
   macro avg       0.92      0.94      0.93       200
weighted avg       0.94      0.94      0.94       200



In [12]:
import joblib

joblib.dump(scaler, "../models/scaler.pkl")

['../models/scaler.pkl']

In [14]:
joblib.dump(stacking_model, "../models/stacking_model.pkl")

['../models/stacking_model.pkl']